# Targeted ablations X1–X4

Plots for the four [targeted ablations](../ABLATIONS.md), each in the layout of the paper's matching appendix figure. Data come from `our-results-folder/ablations/<id>/` and are compared with the matching main-sweep runs in `our-results-folder/data/vision/`. All runs use augmented CIFAR-10 with 20% label noise. Errors are fractions on the clean test set. Every cell has one seed, except X2, which has three.

| Ablation | Question | Paper layout |
| --- | --- | --- |
| X1 · horizon | Are the missing second descent and "more data hurts" region caused by our short training? | Figure 9 (trajectories) + model-wise curves |
| X2 · seeds + ensemble | Is the peak larger than seed variation, and does ensembling help most near the threshold? | Figure 28 |
| X3 · weight decay | Does ℓ2 regularization shrink the peak and the sample-wise effect? | Figures 21 and 11(a) |
| X4 · optimizer | Does epoch-wise double descent survive other optimizers and schedules? | Figures 16 and 18 |

The figure code lives in [`ablation_figures.py`](ablation_figures.py), alongside [`paper_figures.py`](paper_figures.py) for the main figures. Every loader checks the complete width × time grid and the training protocol, and raises an error rather than plotting a partial grid.

In [ ]:
from io import BytesIO
from pathlib import Path
import sys
import matplotlib.pyplot as plt
from IPython.display import Image, display

current = Path.cwd().resolve()
if (current / "our-results-folder" / "ablations").is_dir():
    project_dir = current
elif (current / "Assignment 2 - Group" / "our-results-folder" / "ablations").is_dir():
    project_dir = current / "Assignment 2 - Group"
elif (current.parent / "our-results-folder" / "ablations").is_dir():
    project_dir = current.parent
else:
    raise FileNotFoundError("Run from ATDL-Group-Work, Assignment 2 - Group, or our-plots")
sys.path.insert(0, str(project_dir / "our-plots"))
from ablation_figures import (figure_x1, figure_x2, figure_x3_dynamics, figure_x3_samples, figure_x4,
                              load_x1, load_x2, load_x3, load_x4, save_ablation_figures,
                              x1_endpoints, x3_endpoints)
from paper_figures import style_figure

x1_resnet, x1_cnn, x1_base_resnet, x1_base_cnn = load_x1(project_dir)
x2_summary, x2_baseline = load_x2(project_dir)
x3_runs, x3_baseline = load_x3(project_dir)
x4_arms = load_x4(project_dir)
print("Loaded X1, X2 (predictions checked against recorded test errors), X3, and X4 arms:", ", ".join(x4_arms))


def show_figure(fig: plt.Figure) -> None:
    style_figure(fig)
    buffer = BytesIO()
    fig.savefig(buffer, format="png", dpi=140, bbox_inches="tight")
    display(Image(data=buffer.getvalue()))
    plt.close(fig)

## X1 · training horizon

The left column follows test error over log time for each width, as in paper Figure 9. Solid lines are the ablation runs, extended to 1,200 epochs (ResNet18) or 200,000 steps (CNN). Dashed lines are the main sweep with identical settings, which stops at the dotted line. The right column shows the model-wise curve at the main-sweep horizon and at the extended horizon, with train error dashed. The red line is the minimum observed test error, an oracle early-stopping envelope rather than a validation-selected rule. While the solid and dashed lines overlap, they also show how closely a re-run reproduces the main sweep.

In [ ]:
show_figure(figure_x1(x1_resnet, x1_cnn, x1_base_resnet, x1_base_cnn))

In [ ]:
x1_endpoints(x1_resnet, x1_cnn).round(3)

## X2 · seeds and ensembling

This mirrors paper Figure 28. Blue shows the mean test error of the three independently trained models (seeds 0–2), with a ±1 standard-deviation band. Each seed has its own initialization, data order and label-noise mask. Red shows the plurality-vote ensemble of the same three models; when all three disagree, the lowest class index wins. Grey crosses mark the main sweep's seed-0 run with identical settings. The right panel shows the ensemble gain per width. Loading checks that every saved prediction file reproduces its run's recorded test error exactly.

In [ ]:
show_figure(figure_x2(x2_summary, x2_baseline))

In [ ]:
x2_summary.round(4)

## X3 · weight decay (paper Figure 21 layout)

This uses the full 50,000-example training set. The left panels plot test error against width at every saved step, coloured from early (yellow) to late (purple): without weight decay (main sweep) and with weight decay 5·10⁻⁴. The right column compares test error, train error and test loss at step 50,000. The paper's Figure 21 used 10% noise and 500,000 steps.

In [ ]:
show_figure(figure_x3_dynamics(x3_runs, x3_baseline))

## X3 · weight decay and sample size (paper Figure 11(a) layout)

Rows separate the settings without and with weight decay; columns show test and train error for 12,500 and 50,000 examples after 50,000 SGD steps. The right panel plots test error at 50,000 examples minus test error at 12,500 examples. Values above zero would mark widths where 4× more data hurts.

In [ ]:
show_figure(figure_x3_samples(x3_runs, x3_baseline))

In [ ]:
x3_table = x3_endpoints(x3_runs, x3_baseline)
x3_table.round({column: 3 for column in x3_table.columns if column != "weight_decay"})

## X4 · optimizer and learning-rate schedule (paper Figures 16 and 18 layout)

Test error (top) and train error (bottom) against optimizer iterations for widths 12 and 64. The main-sweep Adam run (LR 10⁻⁴, constant) is the reference. Arms without results are named in the caption.

In [ ]:
show_figure(figure_x4(x4_arms))

**Check before interpreting X4.** Epoch-wise double descent needs the model to keep fitting the noisy labels. The cell below lists each arm's range of train error after 20,000 iterations. A range near zero means training stalled: for example, dynamic drop reduced the learning rate to almost nothing. Such an arm cannot show a second descent.

In [ ]:
import pandas as pd
pd.DataFrame([
    {"arm": label, "model_width": width,
     "train_error_min": run.loc[run["global_step"] >= 20000, "train_error"].min(),
     "train_error_max": run.loc[run["global_step"] >= 20000, "train_error"].max(),
     "final_test_error": run.sort_values("global_step")["test_error"].iloc[-1]}
    for label, frame in x4_arms.items() for width, run in frame.groupby("model_width")
]).round(4)

## Save the figures

This writes the five PNGs to `plots/ablations/`. The same command, run outside the notebook, is in [`../plots/README.md`](../plots/README.md).

In [ ]:
for name, path in save_ablation_figures(project_dir, project_dir / "plots" / "ablations").items():
    print(name, "→", path.relative_to(project_dir))